In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Intel only
# !pip install scikit-learn-intelex
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [3]:
import sys 
import os
sys.path.append('..')
import gc

In [4]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import time

In [5]:
UPSAMPLE_LATER = True 
UPSAMPLE_RATIO = 1

RETRAIN_KNN = True
RETRAIN_TRANSFORM = True

USING_SMOTE = False
FILL_MEAN = True


In [6]:
from utils.check_feature import power_scaler_col

In [7]:
appl_train = pd.read_csv('../data/dseb63_application_train.csv', index_col=0)

# ====== BUREAU ======

bureau = pd.read_parquet('../data/dseb63_bureau_general_v3.parquet')
bureau_columns = bureau.columns

new_bureau_columns = {col: 'BUREAU_' + col for col in bureau_columns if col != 'SK_ID_CURR'} 
bureau.rename(columns=new_bureau_columns, inplace=True)

# ====== PREVIOUS APPLICATION ======

prev_app = pd.read_parquet('../data/prev_app_hanh_v1_1.parquet')
prev_app_columns = prev_app.columns

new_prev_app_columns = {col: 'PREV_APP_' + col for col in prev_app_columns if col != 'SK_ID_CURR'}
prev_app.rename(columns=new_prev_app_columns, inplace=True)

# ====== INSTALLMENTS PAYMENTS ======

installments = pd.read_parquet('../data/dseb63_installment_gb.parquet')
# installments = pd.read_csv('../data/dseb63_installment_gb.csv')
installments_columns = installments.columns

installments.drop(columns= [col for col in installments_columns if 'TARGET' in col], inplace=True)

new_installments_columns = {col: 'INSTALLMENTS_' + col for col in installments_columns if col != 'SK_ID_CURR'}
installments.rename(columns=new_installments_columns, inplace=True)

# ====== CREDIT CARD BALANCE ======

credit_card = pd.read_parquet('../data/dseb63_credit_card_balance_gb.parquet')
credit_card_columns = credit_card.columns

new_credit_card_columns = {col: 'CREDIT_CARD_' + col for col in credit_card_columns if col != 'SK_ID_CURR'}
credit_card.rename(columns=new_credit_card_columns, inplace=True)

# ====== POS CASH BALANCE ======

pos_cash = pd.read_parquet('../data/dseb63_pos_cash_gb.parquet')
# pos_cash = pd.read_csv('../data/dseb63_pos_cash_gb.csv')
pos_cash_columns = pos_cash.columns

new_posh_cash_columns = {col: 'POS_CASH_' + col for col in pos_cash_columns if col != 'SK_ID_CURR'}
pos_cash.rename(columns=new_posh_cash_columns, inplace=True)

# ====== MERGE APP PREV APP ======

app_prev_app = pd.read_parquet('../data/dseb63_app_prev_app_features.parquet')
app_prev_app_columns = app_prev_app.columns

new_app_prev_app_columns = {col: 'APP_PREV_APP_' + col for col in app_prev_app_columns if col != 'SK_ID_CURR'}
app_prev_app.rename(columns=new_app_prev_app_columns, inplace=True)

# ====== MERGE DATA ======

df = appl_train.merge(bureau, on='SK_ID_CURR', how='left')
df = df.merge(installments, on='SK_ID_CURR', how='left')
df = df.merge(pos_cash, on='SK_ID_CURR', how='left')
df = df.merge(credit_card, on='SK_ID_CURR', how='left')
df = df.merge(prev_app, on='SK_ID_CURR', how='left')
# df = df.merge(app_prev_app, on='SK_ID_CURR', how='left')


In [8]:
def RELU(series):
    return series.apply(lambda x: max(0, x))
    

In [9]:
# del bureau, installments, pos_cash, credit_card, prev_app
# gc.collect()

In [10]:
def process_df(df):
    # df = df.copy()
    
    df['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
    df['FONDKAPREMONT_MODE'].fillna('Unknown', inplace=True)
    df['WALLSMATERIAL_MODE'].fillna('Not Specified', inplace=True)
    df['OCCUPATION_TYPE'].replace('IT staff', 'High skill tech staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('Realty agents', 'Sales staff', inplace=True)
    df['OCCUPATION_TYPE'].replace('HR staff', 'Laborers', inplace=True)
    df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)
    df['OCCUPATION_TYPE'].replace(['Cleaning staff', 'Cooking staff', 'Waiters/barmen staff'], 'F&B staff', inplace=True)
    

    df['ORGANIZATION_TYPE'].replace('XNA', 'Unknown', inplace=True)
    df['HOUSETYPE_MODE'].replace(['block of flats', 'specific housing'], 'house', inplace=True)
    
    df['WALLSMATERIAL_MODE'].replace(['Others', 'Mixed', 'Monolithic'], 'Others', inplace=True)
    df['WALLSMATERIAL_MODE'].replace(['Block', 'Stone, brick'], 'Brick', inplace=True)

    map_week_day = {
    'MONDAY': 'week_day',
    'TUESDAY': 'week_day',
    'WEDNESDAY': 'week_day',
    'THURSDAY': 'week_day',
    'FRIDAY': 'week_day',
    'SATURDAY': 'weekend',
    'SUNDAY': 'weekend',
    }
    
    

    df['WEEKDAY_APPR_PROCESS_START'] = df['WEEKDAY_APPR_PROCESS_START'].map(map_week_day)

    map_edu = {
        'Lower secondary': 0,
        'Secondary / secondary special': 1,
        'Incomplete higher': 2,
        'Higher education': 3,
        'Academic degree': 5
    }

    df['NAME_EDUCATION_TYPE'] = df['NAME_EDUCATION_TYPE'].map(map_edu, na_action='ignore').astype(int).fillna(0)

    df['NAME_FAMILY_STATUS'].replace('Unknown', 'Single / not married', inplace=True)
    df['CODE_GENDER'].replace('XNA', 'F', inplace=True)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(365243)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(0, 365243)

    others = df['NAME_INCOME_TYPE'].value_counts().index[4:]
    df['NAME_INCOME_TYPE'].replace(others, 'Others', inplace=True)
    df['NAME_TYPE_SUITE'].fillna('Unaccompanied', inplace=True)

    df['OWN_CAR_AGE'].fillna(-100, inplace=True)
    df['TOTALAREA_MODE'].fillna(0, inplace=True)
    
    df['AGE_INT'] = -df['DAYS_BIRTH'] // 365
    # Steal code
    # df['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan, inplace=True)
    
    def group_organizations(org_type):
        if 'Trade' in org_type:
            return 'Trade'
        elif 'Industry' in org_type:
            return 'Industry'
        elif 'Business' in org_type:
            return 'Business Entity'
        elif 'Transport' in org_type:
            return 'Transport'
        elif 'University' in org_type:
            return 'School'
        elif org_type in ['Police', 'Electricity', 'Culture', 'Religion', 'Telecom', 'Emergency', 'Mobile', 'Postal']:
            return 'Public Sector'

        else:
            return org_type
    
    df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].apply(group_organizations)
    df['NAME_TYPE_SUITE'] = df['NAME_TYPE_SUITE'].replace(['Other_A', 'Other_B'], 'Other')
    
    med_income = df.groupby(['ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE'])['AMT_INCOME_TOTAL'].transform('median')
    med_income2 = df.groupby('ORGANIZATION_TYPE')['AMT_INCOME_TOTAL'].transform('median')
    df['income_ratio'] = df['AMT_INCOME_TOTAL'] / med_income
    df['income_ratio2'] = df['AMT_INCOME_TOTAL'] / med_income2
    df['true_annuity_div_income'] = df['AMT_ANNUITY'] / med_income
    df['true_annuity_div_income2'] = df['AMT_ANNUITY'] / med_income2
    df['true_income_div_totalarea'] = med_income / df['TOTALAREA_MODE'].clip(0.001,1)
    df['true_income_div_totalarea2'] = med_income2 / df['TOTALAREA_MODE'].clip(0.001,1)
    
    df['annuity_income_percentage'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['car_to_birth_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_BIRTH'])
    df['car_to_employ_ratio'] = RELU(df['OWN_CAR_AGE'] / df['DAYS_EMPLOYED'])
    df['children_ratio'] = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS']
    df['credit_to_annuity_ratio'] = df['AMT_CREDIT'] / df['AMT_ANNUITY'] # check
    df['credit_to_goods_ratio'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['credit_to_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    
    
    df['ext_sources_mean'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    df['ext_sources_sum'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].sum(axis=1)
    df['ext_sources_var'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].var(axis=1)
    df['ext_sources_weighted'] = df.EXT_SOURCE_1 * 2 + df.EXT_SOURCE_2 * 3 + df.EXT_SOURCE_3 * 4
    df['EXT_SOURCE_MISSING_VALUES'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isna().sum(axis=1)
    
    df['income_credit_percentage'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    df['income_per_child'] = df['AMT_INCOME_TOTAL'] / (1 + df['CNT_CHILDREN'])
    df['income_per_person'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['phone_to_birth_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH'])
    df['phone_to_employ_ratio'] = RELU(df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_EMPLOYED'] + 1e-5)) * (df['DAYS_EMPLOYED'] < 0) 
    
    df['cnt_non_child'] = df['CNT_FAM_MEMBERS'] - df['CNT_CHILDREN']
    df['child_to_non_child_ratio'] = df['CNT_CHILDREN'] / df['cnt_non_child'] * (df['cnt_non_child'] > 0)
    df['income_per_non_child'] = df['AMT_INCOME_TOTAL'] / df['cnt_non_child']* (df['cnt_non_child'] > 0)
    df['credit_per_person'] = df['AMT_CREDIT'] / df['CNT_FAM_MEMBERS']* (df['cnt_non_child'] > 0)
    df['credit_per_child'] = df['AMT_CREDIT'] / (1 + df['CNT_CHILDREN'])
    df['credit_per_non_child'] = df['AMT_CREDIT'] / df['cnt_non_child']* (df['cnt_non_child'] > 0)

    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['DEBT_BURDEN_PER_WORKING_DAY'] = df['PAYMENT_RATE'] / (df['DAYS_EMPLOYED'] + 1e-5) * (df['DAYS_EMPLOYED'] < 0) 
    df['DEBT_BURDEN_PER_LIFE_DAY'] = df['PAYMENT_RATE'] / df['DAYS_BIRTH']
    df['CREDIT_GOODS_PRICE_RATIO1'] = (df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']) /  df['AMT_GOODS_PRICE']
    df['CREDIT_GOODS_PRICE_RATIO2'] = (df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']) /  df['AMT_CREDIT']
    df['CREDIT_DOWN_PAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']

    df['sin_HOUR_APPR_PROCESS_START'] = np.sin(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df['cos_HOUR_APPR_PROCESS_START'] = np.cos(2 * np.pi * df['HOUR_APPR_PROCESS_START'] / 24)
    df.drop(columns=['HOUR_APPR_PROCESS_START'], inplace=True)

    docs = [f for f in df.columns if 'FLAG_DOC' in f]
    df['NEW_DOC_IND_AVG'] = df[docs].mean(axis=1)
    df['NEW_DOC_IND_STD'] = df[docs].std(axis=1)
    df['NEW_DOC_IND_KURT'] = df[docs].kurtosis(axis=1)
    df['HAS_DOCUMENT'] = df[docs].max(axis=1)
    df['DOCUMENT_COUNT'] = df[docs].sum(axis=1)

    #Drop flag document  
    flag_document = [f for f in df.columns if 'FLAG_DOCUMENT_' in f]
    df.drop(columns=flag_document, inplace=True)
    
    df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'].fillna(0)
    df['LANDAREA_AVG'] = df['LANDAREA_AVG'].fillna(0)
    df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'].fillna(0)
    df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'].fillna(0)
    df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'].fillna(0)

    df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAREA_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAREA_AVG'] / df['LIVINGAREA_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_LANDAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LANDAREA_AVG'].clip(0.05,1) +1e-5) * (df['LANDAREA_AVG'] / df['LANDAREA_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_FLOORSMAX_AVG_AVG'] = (df['AMT_GOODS_PRICE'] / df['FLOORSMAX_AVG'].clip(0.05,1) +1e-5) * (df['FLOORSMAX_AVG'] / df['FLOORSMAX_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAPARTMENTS_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAPARTMENTS_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAPARTMENTS_AVG'] / df['LIVINGAPARTMENTS_AVG']+1e-5)
    df['RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG'] = (df['AMT_GOODS_PRICE'] / df['YEARS_BUILD_AVG'].clip(0.05,1) +1e-5) * (df['YEARS_BUILD_AVG'] / df['YEARS_BUILD_AVG']+1e-5)
    
    
    df['RELIABILITY_IN_CUSTOMER_CITY'] = df['REG_CITY_NOT_LIVE_CITY'] + df['REG_CITY_NOT_WORK_CITY'] + df['REG_REGION_NOT_LIVE_REGION'] + df['REG_REGION_NOT_WORK_REGION'] + df['LIVE_CITY_NOT_WORK_CITY'] + df['LIVE_REGION_NOT_WORK_REGION']
    df['SUM_CONTACTS'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE'] + df['FLAG_CONT_MOBILE'] + df['FLAG_PHONE'] + df['FLAG_EMAIL']

    #Some features from bureau 
    df['TOTAL_ENQUIRIES_CREDIT_BUREAU'] = df[['AMT_REQ_CREDIT_BUREAU_DAY',
                                            'AMT_REQ_CREDIT_BUREAU_HOUR',
                                            'AMT_REQ_CREDIT_BUREAU_WEEK',
                                            'AMT_REQ_CREDIT_BUREAU_MON',
                                            'AMT_REQ_CREDIT_BUREAU_QRT',
                                            'AMT_REQ_CREDIT_BUREAU_YEAR']].sum(axis=1)
    
    
        
    # df['PCTG_ENQUIRIES_HOUR'] = df['AMT_REQ_CREDIT_BUREAU_HOUR'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    # df['PCTG_ENQUIRIES_DAY'] = df['AMT_REQ_CREDIT_BUREAU_DAY'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_WEEK'] = df['AMT_REQ_CREDIT_BUREAU_WEEK'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_MON'] = df['AMT_REQ_CREDIT_BUREAU_MON'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_QRT'] = df['AMT_REQ_CREDIT_BUREAU_QRT'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    df['PCTG_ENQUIRIES_YEAR'] = df['AMT_REQ_CREDIT_BUREAU_YEAR'] / df['TOTAL_ENQUIRIES_CREDIT_BUREAU']

    df.drop(columns=['AMT_REQ_CREDIT_BUREAU_DAY','AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_WEEK'], inplace=True)

    missing_columns = ['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
                   'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
                   'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
                   'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG']

    df['MISSING_GRADINGS'] = df[missing_columns].isna().sum(axis=1)
    # Reliability in customer city or region of residence
    df['RELIABILITY_IN_CUSTOMER_CITY'] = df['REG_CITY_NOT_LIVE_CITY'] + df['REG_CITY_NOT_WORK_CITY'] + df['REG_REGION_NOT_LIVE_REGION'] + df['REG_REGION_NOT_WORK_REGION'] + df['LIVE_CITY_NOT_WORK_CITY'] + df['LIVE_REGION_NOT_WORK_REGION']
    df['SUM_CONTACTS'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE'] + df['FLAG_CONT_MOBILE'] + df['FLAG_PHONE'] + df['FLAG_EMAIL']

    # numerical transformation
    df['DAYS_EMPLOYED'].replace(365243,0, inplace=True)
    df['days_employed_percentage'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    
    
    df['REGION_POPULATION_RELATIVE'] = np.sqrt(df['REGION_POPULATION_RELATIVE'])
    df['APARTMENTS_AVG'] = np.log1p(50 * df['APARTMENTS_AVG'])
    df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'] ** 3
    df['COMMONAREA_AVG'] = df['COMMONAREA_AVG'].clip(0.0001,1) ** (-1/5)
    df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'] ** (1/10)
    df['ENTRANCES_AVG'] = df['ENTRANCES_AVG'] ** (1/3)
    df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'] ** (1/2.5)
    df['FLOORSMIN_AVG'] = df['FLOORSMIN_AVG'] ** (1/2.2)
    df['LANDAREA_AVG'] = df['LANDAREA_AVG'] ** (1/5)
    df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'] ** (1/3)
    df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'] ** (1/3)
    df['NONLIVINGAPARTMENTS_AVG'] = df['NONLIVINGAPARTMENTS_AVG'] ** (1/5)
    df['NONLIVINGAREA_AVG'] = df['NONLIVINGAREA_AVG'] ** (1/3)
    df['OBS_30_CNT_SOCIAL_CIRCLE'] = df['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    df['DEF_30_CNT_SOCIAL_CIRCLE'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    df['OBS_60_CNT_SOCIAL_CIRCLE'] = df['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/5)
    df['DEF_60_CNT_SOCIAL_CIRCLE'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/5)
    
    return df
df = process_df(df)

In [11]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data

In [12]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, OrdinalEncoder, LabelEncoder, MinMaxScaler

In [13]:
# df.to_parquet('../data/df_train.parquet', index=False)

In [14]:
# X = pd.read_parquet('../data/df_train.parquet')
X = df
del df

In [15]:
y = X['TARGET']
X.drop(columns=['TARGET', 'SK_ID_CURR'], inplace=True)


In [16]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246009 entries, 0 to 246008
Columns: 917 entries, NAME_CONTRACT_TYPE to days_employed_percentage
dtypes: float64(876), int64(26), object(15)
memory usage: 1.7+ GB


In [17]:
ordinal_cols = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE', 'CODE_GENDER', 'NAME_CONTRACT_TYPE']
float_cols = []
int_cols = []
cate_cols = []
flag_cols = []
for col in X.columns:
    if col not in ordinal_cols:
        if X[col].dtype == 'float64':
            float_cols.append(col)
        
        elif X[col].dtype == 'int64':

            int_cols.append(col)
        else:
            cate_cols.append(col)

Find suitable column for power transformation

In [18]:
X[float_cols+int_cols].clip(-999999999, 999999999, inplace=True)

In [19]:
from sklearn.model_selection import train_test_split

X_train_, X_val_, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

In [20]:
# TEST

X2 = X.copy()

for col in float_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
    
for col in int_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
for col in cate_cols:
    X2[col].fillna('Unknown', inplace=True)

In [21]:
X2.columns

Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
       'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE',
       ...
       'RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG',
       'RELIABILITY_IN_CUSTOMER_CITY', 'SUM_CONTACTS',
       'TOTAL_ENQUIRIES_CREDIT_BUREAU', 'PCTG_ENQUIRIES_WEEK',
       'PCTG_ENQUIRIES_MON', 'PCTG_ENQUIRIES_QRT', 'PCTG_ENQUIRIES_YEAR',
       'MISSING_GRADINGS', 'days_employed_percentage'],
      dtype='object', length=917)

In [22]:
power_col, standard_col, min_max_col = power_scaler_col(X2[float_cols+int_cols], skewness= 2.5, kurtosis = 10)

100%|██████████| 902/902 [00:23<00:00, 38.47it/s]


In [23]:
len(power_col), len(standard_col), len(min_max_col)

(495, 270, 137)

In [24]:
if FILL_MEAN:
    for col in float_cols:
        X_train_[col].fillna(X_train_[col].mean(), inplace=True)
        X_val_[col].fillna(X_train_[col].mean(), inplace=True)
        
    for col in int_cols:
        X_train_[col].fillna(X_train_[col].mean(), inplace=True)
        X_val_[col].fillna(X_train_[col].mean(), inplace=True)
        
    for col in cate_cols:
        X_train_[col].fillna('Unknown', inplace=True)
        X_val_[col].fillna('Unknown', inplace=True)

else:
    for col in float_cols:
        X_train_[col].fillna(0, inplace=True)
        X_val_[col].fillna(0, inplace=True)
        
    for col in int_cols:
        X_train_[col].fillna(0, inplace=True)
        X_val_[col].fillna(0, inplace=True)
        
    for col in cate_cols:
        X_train_[col].fillna('Unknown', inplace=True)
        X_val_[col].fillna('Unknown', inplace=True)

In [25]:
X_columns = X.columns

### ADD KNN Feature

In [26]:
y = y.to_numpy()
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()

In [27]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

knn_pipeline_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

In [28]:
onehot_transformer = OneHotEncoder(handle_unknown='ignore')
power_transformer = PowerTransformer()
ordinal_transformer = OrdinalEncoder()
scaler_transformer = StandardScaler()
min_max_transformer = MinMaxScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, cate_cols),
        ('power', power_transformer, power_col), # power_transformer
        ('ordinal', ordinal_transformer, ordinal_cols),
        ('scale', scaler_transformer, standard_col),
        ('min_max', scaler_transformer, min_max_col)
        
    ]
)




In [29]:
weak_df_col = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'credit_to_annuity_ratio', 'annuity_income_percentage']
weak_df_col_2 = ['DAYS_ID_PUBLISH', 'DAYS_REGISTRATION', 'days_employed_percentage', 'car_to_birth_ratio', 'NAME_EDUCATION_TYPE', 'AGE_INT']

## Upsample - Downsample

### Manual double

In [30]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import resample

smote = SMOTE(sampling_strategy=UPSAMPLE_RATIO, random_state=42, k_neighbors=5)



In [31]:
def resample_data(X_train, y_train, rate = 0.2):
    if USING_SMOTE:
        return smote.fit_resample(X_train, y_train)
    else:
        posidx = (y_train == 1)
        negidx = (y_train == 0)

        # X_train = np.concatenate([X_train, y_train2], axis=1)
        posidx = np.where(posidx)[0]
        posidx2 = resample(posidx, n_samples=int(len(negidx) * rate), random_state=42)

        idx = np.concatenate([posidx2, np.where(negidx)[0]])

        if isinstance(X_train, pd.DataFrame):
            return X_train.iloc[idx,:], y_train[idx]
        else:
            return X_train[idx,:], y_train[idx]

In [32]:
# KNN -> Transform -> Upsample
if UPSAMPLE_LATER and not USING_SMOTE:

    X_resampled_, y_resampled = resample_data(X_train_, y_train, rate=UPSAMPLE_RATIO)
    print('Fit weak features')
    start = time.time()
    knn_pipeline.fit(X_train_[weak_df_col], y_train) # fit on original data
    knn_pipeline_2.fit(X_train_[weak_df_col_2], y_train) # fit on original data


    weak_feature_train = knn_pipeline.predict_proba(X_resampled_[weak_df_col])[:,1]
    weak_feature_val = knn_pipeline.predict_proba(X_val_[weak_df_col])[:,1]

    weak_feature_train_2 = knn_pipeline_2.predict_proba(X_resampled_[weak_df_col_2])[:,1]
    weak_feature_val_2 = knn_pipeline_2.predict_proba(X_val_[weak_df_col_2])[:,1]

    end = time.time()
    print('Weak features fitted', end-start)

    print('Transforming data')
    start = time.time()
    preprocessor.fit(X_train_) # fit on original data

    X_resampled = preprocessor.transform(X_resampled_)
    X_val = preprocessor.transform(X_val_)
    end = time.time()
    print('Data transformed', end-start)


    X_resampled = np.concatenate([X_resampled, weak_feature_train.reshape(-1,1), weak_feature_train_2.reshape(-1,1)], axis=1)
    X_val = np.concatenate([X_val, weak_feature_val.reshape(-1,1), weak_feature_val_2.reshape(-1,1)], axis=1)
    cat_feature = preprocessor.named_transformers_['onehot'].get_feature_names_out(cate_cols)
    feature_names = np.concatenate([cat_feature, power_col, ordinal_cols, standard_col, min_max_col, ['WEAK_FEATURE', 'WEAK_FEATURE_2']])
    mask_weak_df_col = np.nonzero(np.isin(feature_names, np.array(weak_df_col)))[0]
elif UPSAMPLE_LATER and USING_SMOTE:
    pass 
else:
    raise NotImplementedError('Too lazy to implement')

Fit weak features
Weak features fitted 39.12467312812805
Transforming data
Data transformed 56.631563901901245


In [33]:
X_resampled.shape

(424928, 987)

In [34]:
# X_resampled_.to_parquet('../temp/X_resampled.parquet', index=False)

In [35]:
# X_resampled = pd.DataFrame(X_resampled, columns=feature_names)
# X_val = pd.DataFrame(X_val, columns=feature_names)

# X_resampled['TARGET'] = y_resampled
# X_val['TARGET'] = y_val

# X_resampled.to_parquet('../temp/X_resampled.parquet', index=False)
# X_val.to_parquet('../temp/X_val.parquet', index=False)

In [36]:
# del X_resampled_
# del X_train_

In [37]:

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [38]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

In [39]:
pca = PCA()
pca.fit(X_resampled)

PCA()

In [40]:
explained_variance = pca.explained_variance_ratio_

# Get the absolute values of PCA components (loadings)
feature_contributions = np.abs(pca.components_)

# Aggregate feature importance scores
# Weight each feature's contribution by the variance explained by the component
weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]

# Sum weighted contributions across components
feature_importance = weighted_contributions.sum(axis=0)

# Create a DataFrame for ranking
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)
importance_df.reset_index(drop=True, inplace=True)

In [41]:
importance_df

,Feature,Importance
0,BUREAU_GENERAL_UTILIZATION_RATIO_mean,2.998891e-02
1,PREV_APP_PREV_CODE_REJECT_REASON_HC_SUM,2.952672e-02
2,PREV_APP_PREV_CODE_REJECT_REASON_SCOFR_SUM,2.935959e-02
3,CREDIT_CARD_6_MONTH_CNT_DRAWINGS_ATM_CURRENT_sum,2.876648e-02
4,CREDIT_CARD_12_MONTH_CNT_DRAWINGS_ATM_CURRENT_sum,2.828825e-02
...,...,...
982,PREV_APP_APPROVED_NAME_CONTRACT_STATUS_Unused ...,5.053108e-17
983,PREV_APP_APPROVED_NAME_CONTRACT_TYPE_XNA_SUM,4.862463e-17
984,POS_CASH_NOT_COMPLETED_LONG_CNT_INSTALMENT,2.556887e-17
985,POS_CASH_NOT_COMPLETED_SHORT_CNT_INSTALMENT,2.539177e-17


In [42]:
def select_features_with_pca(X):
    """
    Select features contributing to principal components above a certain threshold.

    Parameters:
    X (numpy array or pandas DataFrame): Input feature matrix.
    threshold (float): Contribution threshold for selecting features (default is 0.1).

    Returns:
    selected_columns (list): List of selected feature names or indices.
    selected_data (DataFrame): Data with selected features.
    """
    # Convert to DataFrame if X is a NumPy array
    # if isinstance(X, np.ndarray):
    #     X = pd.DataFrame(X, columns=[f"Feature{i+1}" for i in range(X.shape[1])])
    
    # Apply PCA
    pca = PCA()
    pca.fit(X)
    
    # Calculate absolute loadings (contributions of features to components)
    explained_variance = pca.explained_variance_ratio_
    feature_contributions = np.abs(pca.components_)
    weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]
    # Identify features exceeding the threshold for any component
    
    feature_importance = weighted_contributions.sum(axis=0)
    
    # Filter the data to keep only selected features
    # selected_data = X[feature_mask, :]
    
    return feature_importance

IF USING PCA

In [43]:
feature_importance = select_features_with_pca(X_resampled)

In [44]:
feature_map = (feature_importance > 0.0005)

In [45]:
# feature_map = np.ones(len(feature_names), dtype=bool)

In [46]:
feature_map.sum()

958

# Modelling

In [47]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer
import numpy as np

class AdaptiveKBinsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_bins=100):
        self.max_bins = max_bins
        self.bin_pipeline_ = None
    
    def fit(self, X, y=None):
        # Create the ColumnTransformer dynamically
        transformers = []
        for col in range(X.shape[1]):
            unique_values = np.unique(X[:, col])
            
            # Adaptive binning logic
            num_bin = max(
                min(
                    self.max_bins, 
                    2 * int(np.sqrt(len(unique_values))) + 1, 
                    len(unique_values)
                ), 
                2
            )
            
            transformers.append((
                f'bin_{col}', 
                KBinsDiscretizer(n_bins=num_bin, encode='ordinal'), 
                [col]
            ))
        
        # Create the column transformer
        self.bin_pipeline_ = ColumnTransformer(transformers, n_jobs=-1)
        
        # Fit the pipeline
        self.bin_pipeline_.fit(X)
        return self
    
    def transform(self, X):
        # Ensure fit has been called
        if self.bin_pipeline_ is None:
            raise ValueError("Transformer has not been fitted yet.")
        
        return self.bin_pipeline_.transform(X)
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)
    
    def get_feature_names_out(self, input_features=None):
        return self.bin_pipeline_.get_feature_names_out(input_features)
    
    def set_params(self, **params):
        if 'max_bins' in params:
            self.max_bins = params['max_bins']
        return self

In [48]:
logistic_model = LogisticRegression(random_state=42, max_iter=10000, class_weight='balanced', C = 0.01)

In [49]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [50]:
from sklearn.metrics import make_scorer, roc_auc_score
def gini_coefficient(y_true, y_pred):
    """
    Calculate the Gini coefficient using predictions and true labels.
    
    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted probabilities.
    
    Returns:
    float: Gini coefficient.
    """
    auc = roc_auc_score(y_true, y_pred)  # AUC calculation
    return 2 * auc - 1  # Gini coefficient

In [51]:
gini_scorer = make_scorer(gini_coefficient, needs_proba=True)

In [52]:
# raise ValueError('Stop here')

In [53]:
class DT_CV(BaseEstimator, TransformerMixin):
    def __init__(self, cv = 5, n_jobs = -1, **kwargs):
        self.cv = cv
        self.bin_pipeline_ = None
        self.base_model = DecisionTreeClassifier(random_state=42, **kwargs)
        self.results = None
    
    def fit(self, X, y=None):
        # Create the ColumnTransformer dynamically
        self.results = cross_validate(self.base_model, X, y, cv=kfold, scoring=gini_scorer, n_jobs=-1,return_estimator=True)
        return self
    

    
    def predict(self, X):
        y_pred = np.zeros((X.shape[0], len(self.results['estimator'])))
        for i, estimator in enumerate(self.results['estimator']):
            y_pred[:,i] = estimator.predict_proba(X)[:,1]
            
        return y_pred.mean(axis=1)
    
    def predict_proba(self, X):
        y_pred = np.zeros((X.shape[0], len(self.results['estimator'])))
        for i, estimator in enumerate(self.results['estimator']):
            y_pred[:,i] = estimator.predict_proba(X)[:,1]
            
        return y_pred.mean(axis=1)
    

    def fit_stacking(self, X, y):
        self.results = dict()
        self.results['estimator'] = []

        kfold = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)
        return_stack = np.zeros((X.shape[0],1))
        for train_idx, val_idx in kfold.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            self.base_model.fit(X_train, y_train)
            y_pred = self.base_model.predict_proba(X_val)[:,1]
            return_stack[val_idx, 0] = y_pred
            self.results['estimator'].append(self.base_model)

        return return_stack
    
    

### Transform first

Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5553
- Gini Score for fold 1: 0.5581
- Gini Score for fold 2: 0.5570
- Gini Score for fold 3: 0.5543
- Gini Score for fold 4: 0.5564

Logistic Regression Cross-Validation Scores:
- Gini Score for fold 0: 0.5531
- Gini Score for fold 1: 0.5532
- Gini Score for fold 2: 0.5542
- Gini Score for fold 3: 0.5553
- Gini Score for fold 4: 0.5532

Stacking Cross-Validation Scores:


Mean Gini Score: 0.5536432676454963

In [54]:
df_test = pd.read_csv('../data/dseb63_application_test.csv',  index_col=0)
# df_test = pd.read_parquet('../data/df_test.parquet')

In [55]:
df_test = df_test.merge(bureau, on='SK_ID_CURR', how='left')
df_test = df_test.merge(installments, on='SK_ID_CURR', how='left')
df_test = df_test.merge(pos_cash, on='SK_ID_CURR', how='left')
df_test = df_test.merge(credit_card, on='SK_ID_CURR', how='left')
df_test = df_test.merge(prev_app, on='SK_ID_CURR', how='left')
# df_test = df_test.merge(app_prev_app, on='SK_ID_CURR', how='left')
df_test = process_df(df_test)

In [56]:
# df_test.to_parquet('../data/df_test.parquet', index=False)

In [57]:
display_missing_data_info(X)

                              Missing Values  Percentage (%)
BUREAU_COUNT_BAD_DEBT                 245993       99.993496
BUREAU_BAD_DEBT_LAST_5_YEAR           245993       99.993496
BUREAU_BAD_DEBT_FINISHED              245993       99.993496
BUREAU_BAD_DEBT_SUM_CREDIT            245993       99.993496
BUREAU_CLOSE_LATENCY_30_DAYS          245252       99.692288
...                                      ...             ...
credit_per_non_child                       1        0.000406
credit_per_person                          1        0.000406
income_per_non_child                       1        0.000406
CNT_FAM_MEMBERS                            1        0.000406
cnt_non_child                              1        0.000406

[846 rows x 2 columns]


,Missing Values,Percentage (%)
BUREAU_COUNT_BAD_DEBT,245993,99.993496
BUREAU_BAD_DEBT_LAST_5_YEAR,245993,99.993496
BUREAU_BAD_DEBT_FINISHED,245993,99.993496
BUREAU_BAD_DEBT_SUM_CREDIT,245993,99.993496
BUREAU_CLOSE_LATENCY_30_DAYS,245252,99.692288
...,...,...
credit_per_non_child,1,0.000406
credit_per_person,1,0.000406
income_per_non_child,1,0.000406
CNT_FAM_MEMBERS,1,0.000406


In [58]:
if FILL_MEAN:
    for col in float_cols:
        if RETRAIN_TRANSFORM:
            X[col].fillna(X[col].mean(), inplace=True)
            df_test[col].fillna(X[col].mean(), inplace=True)
        else:
            X[col].fillna(X_train_[col].mean(), inplace=True)
            df_test[col].fillna(X_train_[col].mean(), inplace=True)
        
    for col in int_cols:
        if RETRAIN_TRANSFORM:
            X[col].fillna(X[col].mean(), inplace=True)
            df_test[col].fillna(X[col].mean(), inplace=True)
        else:
            X[col].fillna(X_train_[col].mean(), inplace=True)
            df_test[col].fillna(X_train_[col].mean(), inplace=True)
        
    for col in cate_cols:
        X[col].fillna('Unknown', inplace=True)
        df_test[col].fillna('Unknown', inplace=True)

else:
    for col in float_cols:
        X[col].fillna(0, inplace=True)
        df_test[col].fillna(0, inplace=True)
        
    for col in int_cols:
        X[col].fillna(0, inplace=True)
        df_test[col].fillna(0, inplace=True)
        
    for col in cate_cols:
        X[col].fillna('Unknown', inplace=True)
        df_test[col].fillna('Unknown', inplace=True)

In [59]:
X_test_ = df_test.drop(columns=['SK_ID_CURR'])
X_test_[float_cols+int_cols] = X_test_[float_cols+int_cols].clip(-999999999, 999999999)

In [60]:
# raise ValueError('Stop here')

In [61]:

# X_resampled_ = pd.read_parquet('../temp/X_resampled.parquet')

## Config retrain stuff

In [62]:


if RETRAIN_KNN:
    
    if not UPSAMPLE_LATER:
        raise NotImplementedError("Too lazy")

    if not USING_SMOTE:
        print('Not using SMOTE')
        if RETRAIN_TRANSFORM:
            print('Retrain Transformer')
            preprocessor.fit(X)

            # Transform X_resampled, add null columns for weak features
            X_resampled = preprocessor.transform(X_resampled_)
            X_resampled = np.concatenate([X_resampled, np.zeros((X_resampled.shape[0],2))], axis = 1)
                
        X_test = preprocessor.transform(X_test_)
        X_val_resampled_, y_val_resampled = resample_data(X_val_, y_val, rate=UPSAMPLE_RATIO/2)
        X_val_resampled = preprocessor.transform(X_val_resampled_)
        # Old KNN pipeline
        knn_pipeline.fit(X[weak_df_col], y)
        knn_pipeline_2.fit(X[weak_df_col_2], y)

        # Update X_resampled
        X_resampled[:, -2] = knn_pipeline.predict_proba(X_resampled_[weak_df_col])[:,1]
        X_resampled[:, -1] = knn_pipeline_2.predict_proba(X_resampled_[weak_df_col_2])[:,1]
        

X_val_resampled = np.concatenate([X_val_resampled, 
                                        knn_pipeline.predict_proba(X_val_resampled_[weak_df_col])[:,1].reshape(-1,1),
                                        knn_pipeline_2.predict_proba(X_val_resampled_[weak_df_col_2])[:,1].reshape(-1,1)
                                        ], axis=1)
     
        
X_test = np.concatenate([X_test, 
                         knn_pipeline.predict_proba(X_test_[weak_df_col])[:,1].reshape(-1,1),
                         knn_pipeline_2.predict_proba(X_test_[weak_df_col_2])[:,1].reshape(-1,1)], axis=1)

X_all_ = np.concatenate([X_resampled, X_val_resampled])
y_all = np.concatenate([y_resampled, y_val_resampled])

Not using SMOTE
Retrain Transformer


In [63]:
y_all.shape

(459841,)

In [64]:
# dt_preprocessor = ColumnTransformer( # Maintain the distribution of the data for Quantile KBinsDiscretizer
#         transformers=[
#             ('onehot', onehot_transformer, cate_cols),
#             ('power', scaler_transformer, power_col), # power_transformer
#             ('ordinal', ordinal_transformer, ordinal_cols),
#             ('scale', scaler_transformer, standard_col),
#             ('min_max', min_max_transformer, min_max_col)
            
#         ]
#     )
  
# X_original = pd.concat([X_resampled_, X_val_resampled_])
# X_original.shape  
    
# dt_preprocessor.fit(X)
# X_dt_ = dt_preprocessor.transform(X)
# X_dt = dt_preprocessor.transform(X_original)
# X_test_dt = dt_preprocessor.transform(X_test_)

# print('Fitting Bins')
# adaptive_kbin = AdaptiveKBinsTransformer(max_bins=100)
# adaptive_kbin.fit(X_dt_)
# X_dt = adaptive_kbin.transform(X_dt)
# X_test_dt = adaptive_kbin.transform(X_test_dt)

# print('Fitting DT')
# dt = DT_CV(cv = 5, max_depth=5, class_weight = 'balanced', min_samples_split=4000, min_samples_leaf=1000, criterion='entropy')
# y_dt_stacking = dt.fit_stacking(X_dt, y_all)
# y_dt_stacking_test = dt.predict_proba(X_test_dt).reshape(-1,1)

# # print('Fitting LR')
# # weak_lr_2 = LR_CV(cv=5, max_iter=1000, class_weight='balanced', C = 0.1)
# # y_lr_stacking = weak_lr_2.fit_stacking(X_original[weak_lr_feat_col], y_all)
# # y_lr_stacking_test = weak_lr_2.predict_proba(X_test_[weak_lr_feat_col]).reshape(-1,1)

# X_all = np.concatenate([X_all_[:, feature_map], y_dt_stacking ], axis=1)
# X_test2 = np.concatenate([X_test[:, feature_map], y_dt_stacking_test], axis=1)

In [65]:
# X_all = np.concatenate([X_all_[:, feature_map], y_dt_stacking ], axis=1)
# X_test2 = np.concatenate([X_test[:, feature_map], y_dt_stacking_test], axis=1)

In [66]:
del X_resampled
del y_resampled
# del X_resampled_
del X_val_resampled
del y_val_resampled

gc.collect()

44

In [67]:
logistic_model = LogisticRegression(random_state=42, max_iter=10000, class_weight='balanced', C = 0.01)

In [68]:
logistic_model.fit(X_all_[:, feature_map], y_all)
y_pred = logistic_model.predict_proba(X_test[:, feature_map])

In [69]:
# logistic_model.fit(X_all, y_all)
# y_pred = logistic_model.predict_proba(X_test2)  

In [70]:
# gini_coefficient(y, y_pred[:, 1])

In [73]:
submission = pd.DataFrame({
    'SK_ID_CURR': df_test['SK_ID_CURR'],
    'TARGET': y_pred[:, 1]
})
submission.to_csv('../submission/dseb63_log_mean_retrain_upsample_1_high_reg.csv', index=False)

In [72]:
ground_truth = pd.read_csv('../temp/target.csv')
submission = submission.merge(ground_truth, on='SK_ID_CURR', how='left')
gini_coefficient(submission['True_Target'], submission['TARGET'])

0.5624711544608043

### With DT
0.5616341552721817

## Without DT
0.5612742520523837